In [2]:
import json
import os
import urllib.error
import urllib.request
from urllib.parse import urlsplit, urlunsplit

from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential
from dotenv import load_dotenv

load_dotenv()


def get_credential():
    try:
        credential = DefaultAzureCredential()
        credential.get_token("https://management.azure.com/.default")
        return credential
    except Exception:
        return InteractiveBrowserCredential()


def get_ml_client():
    subscription_id = os.getenv("SUBSCRIPTION")
    resource_group = os.getenv("RESOURCE_GROUP")
    workspace_name = os.getenv("WS_NAME")

    if not all([subscription_id, resource_group, workspace_name]):
        raise RuntimeError("SUBSCRIPTION, RESOURCE_GROUP, and WS_NAME must be set in the environment.")

    return MLClient(get_credential(), subscription_id, resource_group, workspace_name)


def build_swagger_url(scoring_uri):
    parsed = urlsplit(scoring_uri)
    return urlunsplit((parsed.scheme, parsed.netloc, "/swagger.json", "", ""))


def fetch_swagger(swagger_url, api_key):
    headers = {
        "Accept": "application/json",
        "Authorization": f"Bearer {api_key}",
    }
    request = urllib.request.Request(swagger_url, headers=headers, method="GET")

    with urllib.request.urlopen(request) as response:
        return json.loads(response.read().decode("utf-8"))


endpoint_name = os.getenv("ENDPOINT_NAME", "sklearn-diabetes-05141720427035")
ml_client = get_ml_client()
endpoint = ml_client.online_endpoints.get(endpoint_name)
api_key = os.getenv("API_KEY") or ml_client.online_endpoints.get_keys(endpoint_name).primary_key
swagger_url = build_swagger_url(endpoint.scoring_uri)

print(f"Endpoint: {endpoint.name}")
print(f"Scoring URI: {endpoint.scoring_uri}")
print(f"Swagger URL: {swagger_url}")

try:
    swagger = fetch_swagger(swagger_url, api_key)
    print(json.dumps(swagger, indent=2))
except urllib.error.HTTPError as error:
    print(f"The request failed with status code: {error.code}")
    print(error.info())
    print(error.read().decode("utf-8", "ignore"))


The request failed with status code: 424
server: azureml-frontdoor
date: Mon, 25 May 2026 03:14:17 GMT
content-type: application/json
content-length: 63
x-ms-run-function-failed: False
x-request-id: b2e693a7-d95d-4318-ba7f-cf10b809897a
ms-azureml-model-error-reason: model_error
ms-azureml-model-error-statuscode: 405
azureml-model-deployment: blue
azureml-model-session: blue
connection: close


{"message": "The method is not allowed for the requested URL."}
